# Build KonakolSwaraLLM GGUF (free, on Colab T4)

This notebook takes the **KonakolSwaraLLM** LoRA adapter ([sgattup/KonakolSwaraLLM](https://huggingface.co/sgattup/KonakolSwaraLLM) — a Llama 3.2 3B fine-tune for Carnatic Konakol/Solkattu composition, Apache 2.0), merges it with the base model, and produces a **Q4_K_M GGUF** you can run:

- on your PC with **llama.cpp** (`llama-server`), **LM Studio**, or **Ollama**;
- on Android with the llama.cpp Android build or Termux;
- fully offline, free, no API keys.

The Korvai Composer apps (web + Android) then call it at `http://127.0.0.1:8080` as their optional AI selector. **Every AI proposal is still re-validated by the deterministic tala engine** — the model's own card disclaims beat-count precision, so the engine never trusts its arithmetic.

Runtime: **GPU T4** (free tier) · ~20 min · nothing leaves your machine except the HF downloads.

In [ ]:
# 1) Install deps (pinned, free-tier friendly)
!pip -q install "transformers>=4.45" "peft>=0.13" "accelerate>=0.34" "torch" sentencepiece protobuf huggingface_hub

## 2) Merge the LoRA adapter into the base model

The adapter repo ships only the ~97 MB LoRA weights (`adapter_model.safetensors`), so we pull the unquantized base mirror (`unsloth/llama-3.2-3b-instruct`, ungated) in bf16 and merge.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "unsloth/llama-3.2-3b-instruct"          # ungated mirror of meta-llama/Llama-3.2-3B-Instruct
ADAPTER = "sgattup/KonakolSwaraLLM"
MERGED_DIR = "/content/konakolswara-merged"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)   # adapter repo carries the tokenizer files
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map={"": 0}
)
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("merged OK →", MERGED_DIR)

## 3) Get llama.cpp and convert to GGUF

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile /content/konakolswara-f16.gguf --outtype f16

## 4) Quantize to Q4_K_M (~2.0 GB — fits phones comfortably)

In [ ]:
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 2 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/konakolswara-f16.gguf /content/konakolswara-llm-Q4_K_M.gguf Q4_K_M
!ls -lh /content/konakolswara-llm-Q4_K_M.gguf

## 5) Smoke-test the model with the exact prompt format the apps use

In [ ]:
PROMPT = '''You are an expert in Carnatic classical music, specializing in Konakol (Solkattu) — the vocal recitation of rhythmic syllables — and Swara sequence composition. You can explain Tala theory, compose creative Konakol patterns, generate melodic Swara sequences, and teach the grammar of South Indian rhythm and melody.

### Question:
Context: Adi tala, Chaturasra nadai, 2 kalai.

Available rhythm cells (matra counts are exact):
  ta ka di mi = 4 matras (core, square, difficulty 2)
  ta ki ta = 3 matras (core, flowing, difficulty 2)
  ta ka ta ki ta = 5 matras (core, square, difficulty 3)
  tha ka dhi mi = 4 matras (core, flowing, difficulty 2)

Task: build a korvai by choosing cells for each slot. Constraints:
  Slot A: exactly 4 matras.
  Slot B: exactly 5 matras.

RULES (obey exactly):
1. For each slot, output ONE LINE: "SLOT <id>: cell1 + cell2 + ..." using the cell notations exactly as listed.
2. The matra counts of the cells on each line MUST sum to exactly the slot's matras.
3. Use only the listed cells. No new syllables. No arithmetic in the answer.
4. Output the slot lines and nothing else.

### Answer:
'''
# quick greedy check on GPU with transformers (before packaging)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MERGED_DIR)
m = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16, device_map={"": 0})
ids = tok(PROMPT, return_tensors="pt").to("cuda")
out = m.generate(**ids, max_new_tokens=120, temperature=0.7, do_sample=True)
print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))
del m
import gc, torch; gc.collect(); torch.cuda.empty_cache()

## 6) Download the GGUF (or push it to your HF repo)

Then serve it — see `MODEL_SETUP.md` in the Korvai Composer repo:

```bash
# llama.cpp (any platform)
llama-server -m konakolswara-llm-Q4_K_M.gguf --host 127.0.0.1 --port 8080

# Ollama
ollama create konakolswara-llm -f Modelfile   # FROM ./konakolswara-llm-Q4_K_M.gguf
ollama serve                                    # OpenAI-compatible at :11434

# LM Studio: load the GGUF, start the local server (default :1234)
```

In the app's AI panel: Base URL `http://127.0.0.1:8080` (llama.cpp) or `http://127.0.0.1:11434` (Ollama, API = OpenAI-compatible), model `konakolswara-llm`.

In [ ]:
from google.colab import files
files.download('/content/konakolswara-llm-Q4_K_M.gguf')

# or push to your own Hugging Face repo:
# from huggingface_hub import HfApi
# HfApi().upload_file(path_or_fileobj="/content/konakolswara-llm-Q4_K_M.gguf",
#                     repo_id="YOUR_USERNAME/konakolswara-llm-gguf",
#                     repo_type="model", path_in_repo="konakolswara-llm-Q4_K_M.gguf")